In [1]:
import jax
jax.print_environment_info()

jax:    0.4.26
jaxlib: 0.4.26
numpy:  1.26.4
python: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:50:58) [GCC 12.3.0]
jax.devices (1 total, 1 local): [cuda(id=0)]
process_count: 1
platform: uname_result(system='Linux', node='r805u30n01.grace.ycrc.yale.edu', release='4.18.0-477.75.1.el8_8.x86_64', version='#1 SMP Mon Sep 30 16:14:54 EDT 2024', machine='x86_64')


$ nvidia-smi
Fri Mar 28 10:25:03 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.42.06              Driver Version: 555.42.06      CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|========================

In [2]:
import matplotlib as mpl
import matplotlib.pyplot as plt

plt.rc("text", usetex=True)
plt.rc("font", **{"family": "DejaVu Sans", "size": 20})
plt.rc("axes", labelsize=16)
plt.rc("xtick", labelsize=16)
plt.rc("ytick", labelsize=16)

from cycler import cycler
from matplotlib import cm
from matplotlib import colors as mc
from matplotlib.colors import LogNorm, PowerNorm
from matplotlib.collections import PolyCollection
from mpl_toolkits.mplot3d import Axes3D

from matplotlib.patches import ConnectionPatch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.patches as patches

%matplotlib inline
# %matplotlib widget

In [3]:
from pmwd import (Configuration, Cosmology, SimpleLCDM, 
                    particles, boltzmann, linear_power, growth, 
                    white_noise, linear_modes, 
                    lpt, nbody, scatter)

from pmwd.vis_util import simshow

import readgadget
import readfof
import redshift_space_library as RSL
import MAS_library as MASL
import smoothing_library as SL
import Pk_library as PKL
import cosmology_library as CL

import numpy as np
import pandas as pd

In [4]:
from pmwd.gravity import laplace, neg_grad
from pmwd.pm_util import fftfreq, fftfwd, fftinv

In [14]:
Configuration?

Init signature:
Configuration(
    ptcl_spacing: float,
    ptcl_grid_shape: Tuple[int, ...],
    mesh_shape: Union[float, Tuple[int, ...]] = 1,
    cosmo_dtype: Union[str, type[Any], numpy.dtype, jax._src.typing.SupportsDType] = <class 'jax.numpy.float64'>,
    pmid_dtype: Union[str, type[Any], numpy.dtype, jax._src.typing.SupportsDType] = <class 'jax.numpy.int16'>,
    float_dtype: Union[str, type[Any], numpy.dtype, jax._src.typing.SupportsDType] = <class 'jax.numpy.float32'>,
    k_pivot_Mpc: float = 0.05,
    T_cmb: float = 2.7255,
    M: float = 1.98847e+40,
    L: float = 3.0856775815e+22,
    T: float = 3.0856775815e+17,
    transfer_fit: bool = True,
    transfer_fit_nowiggle: bool = False,
    transfer_lgk_min: float = -4,
    transfer_lgk_max: float = 3,
    transfer_lgk_maxstep: float = 0.0078125,
    growth_rtol: Optional[float] = None,
    growth_atol: Optional[float] = None,
    growth_inistep: Union[float, NoneType, Tuple[Optional[float], Optional[float]]] = (1, None),
 

### --- you can change `a_start=1/50` to `a_start=1` for the Zeldovich simulation up to redshift zero

In [5]:
BoxSize = 1000.0
grid    = 256

if jax.default_backend() == 'gpu':
    ptcl_spacing = BoxSize/grid  # Lagrangian space Cartesian particle grid spacing, in Mpc/h by default
    ptcl_grid_shape = (grid,) * 3
else:
    ptcl_spacing = BoxSize/64
    ptcl_grid_shape = (64,) * 3

conf = Configuration(ptcl_spacing, ptcl_grid_shape, mesh_shape=1, lpt_order=1, a_start=1/50)  # 1x mesh shape
print(conf)  # with other default parameters

Configuration(ptcl_spacing=3.90625,
              ptcl_grid_shape=(256, 256, 256),
              mesh_shape=(256, 256, 256),
              cosmo_dtype=dtype('float64'),
              pmid_dtype=dtype('int16'),
              float_dtype=dtype('float32'),
              k_pivot_Mpc=0.05,
              T_cmb=2.7255,
              M=1.98847e+40,
              L=3.0856775815e+22,
              T=3.0856775815e+17,
              transfer_fit=True,
              transfer_fit_nowiggle=False,
              transfer_lgk_min=-4,
              transfer_lgk_max=3,
              transfer_lgk_maxstep=0.0078125,
              growth_rtol=1.4901161193847656e-08,
              growth_atol=1.4901161193847656e-08,
              growth_inistep=(1, None),
              lpt_order=1,
              a_start=0.02,
              a_stop=1,
              a_lpt_maxstep=0.0078125,
              a_nbody_maxstep=0.015625,
              symp_splits=((0, 0.5), (1, 0.5)),
              chunk_size=16777216)


In [6]:
print(f'Simulating {conf.ptcl_num} particles with a {conf.mesh_shape} mesh for {conf.a_nbody_num} time steps.')

Simulating 16777216 particles with a (256, 256, 256) mesh for 63 time steps.


In [8]:
cosmo = SimpleLCDM(conf)
print(cosmo)

Cosmology(A_s_1e9=Array(2., dtype=float64),
          n_s=Array(0.96, dtype=float64),
          Omega_m=Array(0.3, dtype=float64),
          Omega_b=Array(0.05, dtype=float64),
          h=Array(0.7, dtype=float64),
          Omega_k_=None,
          w_0_=None,
          w_a_=None,
          transfer=None,
          growth=None,
          varlin=None)


In [9]:
seed  = 123
modes = white_noise(seed, conf, real=True)
cosmo = boltzmann(cosmo, conf)

In [10]:
modes = linear_modes(modes, cosmo, conf)
ptcl, obsvbl = lpt(modes, cosmo, conf)

In [15]:
ptcl.pos()

Array([[ 0.025, -0.049,  0.02 ],
       [ 0.05 , -0.079, -0.18 ],
       ...,
       [-0.04 , -0.237,  0.168],
       [ 0.04 , -0.158,  0.021]], dtype=float32)

In [16]:
ptcl, obsvbl = nbody(ptcl, obsvbl, cosmo, conf)

In [19]:
ptcl.pos()

Array([[6.628e-01, 9.932e+02, 9.977e+02],
       [1.015e+00, 9.925e+02, 9.974e+02],
       ...,
       [9.988e+02, 9.838e+02, 9.979e+02],
       [9.992e+02, 9.853e+02, 9.978e+02]], dtype=float64)

In [20]:
dens = scatter(ptcl, conf)

In [22]:
dens.shape

(256, 256, 256)

In [74]:
def pot(modes, cosmo, conf):
    modes /= conf.ptcl_cell_vol  # remove volume factor first for convenience
    kvec = fftfreq(conf.ptcl_grid_shape, conf.ptcl_spacing, dtype=conf.float_dtype)
    pot  = laplace(kvec, modes, cosmo)
    pot_real = fftinv(pot, shape=conf.ptcl_grid_shape)
    pot_real = pot_real.astype(conf.float_dtype)  # no jnp.complex32
    return pot_real

In [75]:
pot = pot(modes, cosmo, conf)

In [77]:
pot.shape

(256, 256, 256)

In [59]:
ptcl.disp

Array([[ 0.025, -0.049,  0.02 ],
       [ 0.05 , -0.079, -0.18 ],
       ...,
       [-0.04 , -0.237,  0.168],
       [ 0.04 , -0.158,  0.021]], dtype=float32)

In [60]:
np.sqrt(np.sum(ptcl.disp**2, axis=1)).max()

0.9126068

In [67]:
ptcl.pmid

Array([[  0,   0,   0],
       [  0,   0,   1],
       ...,
       [255, 255, 254],
       [255, 255, 255]], dtype=int16)

In [68]:
pos = ptcl.pos()